<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-09/AI_Modul_9.1_Praktikum_Konsep_Deep_Learning.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 9.1: Praktikum Konsep Deep Learning
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Memahami perbedaan mendasar antara representasi linier dangkal (*shallow*) dan representasi hierarkis (*deep*).
2. Mengimplementasikan ekstraksi fitur hierarkis bertingkat menggunakan aljabar matriks NumPy murni.
3. Mendemonstrasikan Teorema Aproksimasi Universal untuk mendekati fungsi non-linier kompleks sensor tanaman.
4. Menganalisis efisiensi komputasi dan perilaku aproksimasi pada variasi kapasitas lapisan tersembunyi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'NumPy Version: {np.__version__}')


## 1. Simulasi Fitur Sensor Tanaman dan Ekstraksi Hierarkis
Membuat representasi sensor 8 kanal (pantulan spektral kanopi kelapa sawit) dan mengekstraksi representasi fitur laten melalui transformasi bertingkat.

In [ ]:
np.random.seed(42)
n_samples = 100
n_channels = 8  # 8 panjang gelombang spektral

# Data spektral sintetik daun sawit [0.0 - 1.0]
X_raw = np.random.uniform(0.1, 0.9, size=(n_samples, n_channels))

# Definisi bobot 2 lapisan ekstraksi fitur
W1 = np.random.randn(16, n_channels) * np.sqrt(2.0 / n_channels)
b1 = np.zeros((1, 16))
W2 = np.random.randn(8, 16) * np.sqrt(2.0 / 16)
b2 = np.zeros((1, 8))

# Forward pass ekstraksi fitur hierarkis
def relu(z):
    return np.maximum(0, z)

H1 = relu(np.dot(X_raw, W1.T) + b1)  # Fitur representasi rendah
H2 = relu(np.dot(H1, W2.T) + b2)     # Fitur representasi tinggi

print('Dimensi data masukan mentah :', X_raw.shape)
print('Dimensi representasi lapisan 1:', H1.shape)
print('Dimensi representasi lapisan 2:', H2.shape)


## 2. Uji Teorema Aproksimasi Universal (Universal Approximation Theorem)
Mendemonstrasikan kemampuan jaringan saraf 1 lapisan tersembunyi dengan aktivasi non-linier dalam mengaproksimasi fungsi respon klorofil non-linier.

In [ ]:
# Membuat fungsi non-linier target agronomis
X_val = np.linspace(-3, 3, 200).reshape(-1, 1)
y_true = np.sin(2.5 * X_val) + 0.4 * np.cos(X_val) + 0.15 * X_val**2

class UniversalApproximator:
    def __init__(self, n_hidden=32, lr=0.03):
        np.random.seed(42)
        self.W1 = np.random.randn(n_hidden, 1) * 0.5
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(1, n_hidden) * 0.5
        self.b2 = np.zeros((1, 1))
        self.lr = lr
        
    def fit(self, X, y, epochs=1500):
        m = X.shape[0]
        history = []
        for ep in range(epochs):
            # Forward
            Z1 = np.dot(X, self.W1.T) + self.b1
            A1 = np.tanh(Z1)
            y_hat = np.dot(A1, self.W2.T) + self.b2
            
            loss = np.mean((y_hat - y)**2)
            history.append(loss)
            
            # Backward
            dy = 2 * (y_hat - y) / m
            dW2 = np.dot(dy.T, A1)
            db2 = np.sum(dy, axis=0, keepdims=True)
            
            dA1 = np.dot(dy, self.W2)
            dZ1 = dA1 * (1 - A1**2)
            dW1 = np.dot(dZ1.T, X)
            db1 = np.sum(dZ1, axis=0, keepdims=True)
            
            # Gradient update
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1
        return history
        
    def predict(self, X):
        Z1 = np.dot(X, self.W1.T) + self.b1
        A1 = np.tanh(Z1)
        return np.dot(A1, self.W2.T) + self.b2

model = UniversalApproximator(n_hidden=48, lr=0.03)
loss_history = model.fit(X_val, y_true, epochs=2000)
y_pred = model.predict(X_val)
rmse = np.sqrt(np.mean((y_pred - y_true)**2))
print(f'Training Selesai! Final RMSE: {rmse:.4f}')


## 3. Visualisasi Hasil Aproksimasi dan Kurva Konvergensi
Menampilkan hasil kurva estimasi model neural network dibandingkan dengan kurva non-linier target aktual.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(X_val, y_true, label='Fungsi Aktual Tanaman (Ground Truth)', color='black', linewidth=2.5)
ax1.plot(X_val, y_pred, label=f'Aproksimasi Jaringan Saraf (RMSE: {rmse:.4f})', color='crimson', linestyle='--', linewidth=2)
ax1.set_title('Aproksimasi Fungsi Non-Linier Sensor Kanopi', fontsize=12, fontweight='bold')
ax1.set_xlabel('Variabel Prediktor Terstandarisasi', fontsize=10)
ax1.set_ylabel('Respons Klorofil', fontsize=10)
ax1.legend()

ax2.plot(loss_history, color='teal', linewidth=2)
ax2.set_title('Dinamika Konvergensi Loss (MSE)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch Iterasi', fontsize=10)
ax2.set_ylabel('Mean Squared Error', fontsize=10)

plt.tight_layout()
plt.savefig('praktikum_aproksimasi_9_1.png', dpi=150)
plt.show()
